# Metrics Demo: CREST Metrics Module

The `Metrics` class (`crest/utils/Metrics.py`) provides Earth-system-focused evaluation
metrics that can be used directly on arrays **and** wired into CREST models during
compilation. It supports:

- **Built-in metrics** — NSE, KGE, RMSE, Pearson *r*, MSE, $\alpha$-NSE, $\beta$-NSE, and more
- **Custom metrics** — any callable with the signature `(y_true, y_pred) → scalar`
- **Keras subclasses** — drop-in `keras.metrics.Metric` wrappers for use in `compile`
- **JSON serialization** — save and restore a metric set across sessions

This notebook covers:

1. **The `Metrics` class** — instantiation, `.all`, and `.customs` properties
2. **Built-in metrics** — calling domain metrics on synthetic data
3. **Custom metrics** — registering and using a user-defined function
4. **Keras-compatible subclasses** — `Metrics.NSE`, `Metrics.MSE`, *etc.*
5. **Metrics in a CREST Model** — attaching `model.metrics` and compiling
6. **Model evaluation** — running `model.evaluate` with a `Batcher`
7. **Serialization** — `to_json` / `from_json` and round-trip file I/O

In [ ]:
# Allow crest to be imported
%cd -q ..

## 1. The Metrics Class

`Metrics` lives in `crest/utils/Metrics.py` and is automatically attached to every
`Model` instance as `model.metrics`. You can also use it as a standalone object
without constructing a model at all.

When instantiated, `Metrics.__init__` calls `register` for every built-in metric and
stores the result in an internal `handlers` defaultdict keyed by metric name. The
complete built-in list is snapshotted in `m.defaults` so that `m.customs` can later
return only user-added entries.

See `tests/utils/test_Metrics.py` for the full test suite covering each metric and
each Keras subclass.

In [ ]:
from crest.utils import Metrics

m = Metrics()

print("All registered metrics:")
for name in m.all:
    print(" ", name)

print("\nCustom metrics (user-added):", m.customs)

## 2. Built-in Metrics

All built-in metrics accept TensorFlow tensors or numpy arrays as `(y_true, y_pred)`
and return a scalar (or tensor). The table below summarizes the domain-specific ones:

| Method | Full name | Perfect score |
|---|---|---|
| `nse` | Nash-Sutcliffe Efficiency | 1 |
| `kge` | Kling-Gupta Efficiency | 1 |
| `unbiased_rmse` | Root Mean Squared Error | 0 |
| `mse` | Mean Squared Error | 0 |
| `pearsonr` | Pearson correlation coefficient | 1 |
| `alpha_nse` | std(pred) / std(obs) | 1 |
| `beta_nse` | (mean(pred) − mean(obs)) / std(obs) | 0 |
| `relative_error` | Mean relative error (NaN-safe) | 0 |
| `nse_log` | NSE in log space (low-flow sensitive) | 1 |
| `metric_entropy` | Normalized Shannon entropy pair | — |

In this example we generate a short synthetic time series with Gaussian noise to demonstrate each.

In [ ]:
import numpy as np
import tensorflow as tf

rng = np.random.default_rng(42)
y_true = rng.uniform(0.1, 10.0, size=365).astype("float32")
y_pred = (y_true + rng.normal(0, 0.5, size=365)).astype("float32")

y_true_tf = tf.constant(y_true)
y_pred_tf = tf.constant(y_pred)

print(f"y_true: min={y_true.min():.2f}  max={y_true.max():.2f}  mean={y_true.mean():.2f}")
print(f"y_pred: min={y_pred.min():.2f}  max={y_pred.max():.2f}  mean={y_pred.mean():.2f}")

In [ ]:
print(f"NSE:            {m.nse(y_true_tf, y_pred_tf):.4f}   (1 = perfect)")
print(f"KGE:            {m.kge(y_true_tf, y_pred_tf):.4f}   (1 = perfect)")
print(f"Unbiased RMSE:  {m.unbiased_rmse(y_true_tf, y_pred_tf):.4f}")
print(f"MSE:            {m.mse(y_true_tf, y_pred_tf).numpy():.4f}")
print(f"Pearson r:      {m.pearsonr(y_true_tf, y_pred_tf):.4f}")
print(f"Alpha-NSE:      {m.alpha_nse(y_true_tf, y_pred_tf):.4f}   (1 = perfect)")
print(f"Beta-NSE:       {m.beta_nse(y_true_tf, y_pred_tf):.4f}   (0 = perfect)")
print(f"Relative error: {m.relative_error(y_true_tf, y_pred_tf).numpy():.4f}")

## 3. Custom Metrics

Any callable with the signature `(y_true, y_pred) → scalar` can be registered via
`register(name, callback)`. Once registered it:

- appears in `m.all` and `m.customs` (but not `m.defaults`)
- is retrievable by name with `m.get_handler(name)`
- survives JSON serialization

Custom metrics can also be passed directly to `model.metrics.register` and will be
saved with `model.save()` — see `crest/model/Model.py:save`.

In [ ]:
def mean_abs_error(y_true, y_pred):
    return tf.reduce_mean(tf.abs(y_true - y_pred))

m.register("mae", mean_abs_error)

print("Built-in defaults (first 3):", m.defaults[:3], "...")
print("Custom metrics:             ", m.customs)

handler = m.get_handler("mae")
print(f"\nMAE via get_handler: {handler(y_true_tf, y_pred_tf).numpy():.4f}")

## 4. Keras-Compatible Subclasses

`Metrics` exposes inner `keras.metrics.Metric` subclasses — e.g. `Metrics.NSE`,
`Metrics.MSE`, `Metrics.KGE`, `Metrics.UnbiasedRMSE`, `Metrics.Pearson` — that
implement the standard `update_state` / `result` / `reset_states` interface.

These can be passed to `keras.Model.compile(metrics=[...])` directly, just like the
built-in Keras metrics. Each subclass delegates to the corresponding instance method on
a fresh `Metrics()` object so the implementation stays in one place.

The full subclass surface is exercised by the `test_*_subclass` tests in
`tests/utils/test_Metrics.py`.

In [ ]:
nse_metric  = Metrics.NSE()
mse_metric  = Metrics.MSE()
rmse_metric = Metrics.UnbiasedRMSE()
kge_metric  = Metrics.KGE()

for metric, label in [
    (nse_metric,  "NSE          "),
    (mse_metric,  "MSE          "),
    (rmse_metric, "Unbiased RMSE"),
    (kge_metric,  "KGE          "),
]:
    metric.update_state(y_true_tf, y_pred_tf)
    print(f"{label}: {metric.result()}")

## 5. Metrics in a CREST Model

`Model.__init__` (`crest/model/Model.py`) attaches a `Metrics` instance as
`model.metrics` (plural). Custom metrics registered on `model.metrics` survive
`model.save()` / `Model.load()` because `save` calls `model.metrics.to_json()` and
`load` calls `Metrics.from_json()`.

Below we build a minimal one-layer regression model with `Node` (`crest/model/Node.py`)
and `StructuredDataset` (`crest/data/loading/StructuredDataset.py`), register a custom
metric on `model.metrics`, then compile and fit.

**Note:** `model.metrics` is the CREST `Metrics` registry and is distinct from the
Keras compile-time `metrics` list passed to `model.compile()`. To use a CREST metric
as a Keras compile metric, pass the corresponding subclass (e.g. `Metrics.NSE()`) in
the `metrics` kwarg of `compile`.

In [ ]:
from tensorflow.keras.layers import Dense
from tensorflow import TensorSpec
from crest import Node
from crest.model.Model import Model
from crest.data.loading import StructuredDataset
from crest.data import Batcher

# Synthetic linear regression: y = 2x + small noise
rng2 = np.random.default_rng(0)
x_data = rng2.uniform(0, 1, size=(500, 1)).astype("float32")
y_data = (2 * x_data + rng2.normal(0, 0.05, size=(500, 1))).astype("float32")

import matplotlib.pyplot as plt
_ = plt.scatter(x_data, y_data)
_ = plt.title(f'y=2x+noise')

In [ ]:
ds = StructuredDataset(x_data, y_data, labels=["x", "y"])

dense = Dense(1)
htg = Node(
    node=lambda X: {"y": dense(X["x"])},
    name="linear",
    inputs={"x": TensorSpec(shape=[None, 1])},
    outputs={"y": TensorSpec(shape=[None, 1])},
)

model = Model(htg)
print("model.metrics defaults (first 3):", model.metrics.all[:3], "...")

In [ ]:
# Register the custom metric defined in §3 with the model
model.metrics.register("mae", mean_abs_error)
print("Custom metrics on model:", model.metrics.customs)

batcher_train = Batcher(ds, **{
    "batch_size": 200,
    "features": [["x"], ["y"]],
    "repeat": True,   # Yield indefinitely
    "shuffle": False, # Turn shuffle off for prediction
    "workers": 0,     # Number of background processes
})

model.compile(optimizer="adam", loss="mse")
model.fit(batcher_train, steps_per_epoch=20, epochs=3, verbose=1)
batcher_train.close()

## 6. Model Evaluation

`model.evaluate` (`crest/model/Model.py:evaluate`) accepts a `Batcher` directly. When
passed a `Batcher` it calls `iter(batcher)` internally — the Batcher starts its
generator lazily via `__iter__`, so no explicit context manager is required.

**Important — always pass `steps` when using `repeat=False`.** because Keras's internal
`EpochIterator` cannot determine the length of a plain Python iterator in advance and 
enters an open-ended loop until `StopIteration` is raised.  Passing 
`steps = len(dataset) // batch_size` tells Keras exactly how many batches to consume, 
so it stops cleanly before the iterator raises `StopIteration`.

The returned dict contains the Keras compile-time loss (and any metrics passed to
`compile`). To apply a CREST `Metrics` function post-hoc, call it directly on
`y_true` / `y_pred` arrays, as shown in `tests/utils/test_Metrics.py`.

In [ ]:
batch_size = 200
batcher_eval = Batcher(ds, **{
    "batch_size": batch_size,  # 200 samples in each batch
    "features": [["x"], ["y"]],
    "repeat": False,           # Exhaust after one pass
    "shuffle": False,
    "workers": 0,
})

# Pass `steps` explicitly so Keras knows when to stop cleanly.
# Without it, Keras iterates until StopIteration and emits a spurious
# "Your input ran out of data" warning from epoch_iterator.py, because it
# cannot infer the length of a plain Python iterator in advance.
n_steps = len(x_data) // batch_size  # 500 // 200 = 2
results = model.evaluate(batcher_eval, steps=n_steps, verbose=0, return_dict=True)
print("Evaluation results:", results)
batcher_eval.close()

# Apply a CREST metric to the raw predictions.
# Pass input as a named dict to match the model's input structure.
y_pred_model = model({"x": x_data})["y"].numpy().flatten()
y_true_flat  = y_data.flatten()

print(f"\nCREST metrics on full dataset:")
print(f"  NSE:  {model.metrics.nse(y_true_flat, y_pred_model):.4f}")
print(f"  MAE:  {model.metrics.get_handler('mae')(y_true_flat, y_pred_model).numpy():.4f}")

## 7. Serialization

`Metrics.to_json()` encodes the full handler dict — both built-ins and any registered
custom metrics — using `jsonpickle`. `Metrics.from_json(json_str)` reconstructs a
fresh `Metrics()` and re-registers any entries that are not already in `.defaults`
(built-ins are always re-created by `__init__`, so they are never double-registered).

This round-trip is covered by `test_save_load`, `test_save_load_file`, and the
`_complex` variants in `tests/utils/test_Metrics.py`.

`Model.save` (`crest/model/Model.py:save`) calls `self.metrics.to_json()` internally,
so custom metrics registered on `model.metrics` are automatically included in the
saved `.crest` archive.

In [ ]:
# In-memory round-trip
saved = m.to_json()
m2 = Metrics.from_json(saved)

print("Custom metrics in restored object:", m2.customs)

handler2 = m2.get_handler("mae")
print(f"MAE (original):  {m.get_handler('mae')(y_true_tf, y_pred_tf).numpy():.4f}")
print(f"MAE (restored):  {handler2(y_true_tf, y_pred_tf).numpy():.4f}")
print(f"NSE matches:     {m.nse(y_true_tf, y_pred_tf) == m2.nse(y_true_tf, y_pred_tf)}")

In [ ]:
import os

# File round-trip
with open("saved_metrics.json", "w") as f:
    f.write(saved)

with open("saved_metrics.json", "r") as f:
    m3 = Metrics.from_json(f.read())

os.remove("saved_metrics.json")

print(f"NSE (restored from file): {m3.nse(y_true_tf, y_pred_tf):.4f}")
print("File round-trip complete.")